In [1]:
import torch
import math
from ot.batch import solve_gromov_batch

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 0) 설정 (네 dist_sq 규모에 맞춤)
# =========================
B = 4
M = 16
N = 8
K = 8
D = 768

alpha = 0.5
reg = 0.05
soft_tau = 0.05

# 네 로그 dist_sq mean~24042 기준으로 src/tgt base 분산 맞추기
# E||x-y||^2 ≈ 2*D*s^2
s = math.sqrt(24042.0 / (2.0 * D))  # ~3.95

# 후보 간 dist_sq 차이를 "수천 단위"로 강제로 만들기
gap_step = 2000.0   # 후보 간 dist_sq 차이 step (원하면 3000으로)
mstar_adv = 2000.0  # 정답 m*를 더 가깝게 만드는 이득(= dist_sq 기준)

# =========================
# 1) src / tgt_base 생성
# =========================
src_feat = torch.randn(B, N, D, device=device) * s
tgt_base = torch.randn(B, M, K, D, device=device) * s

# 샘플별 정답 후보
m_star = torch.randint(0, M, (B,), device=device)

# =========================
# 2) 후보별 "고정 방향" + "고정 에너지" shift로 dist_sq gap 강제
#    - 후보 m마다 unit 방향 U[m] 고정
#    - 후보 m마다 shift norm^2 = delta[m] (=> dist_sq 평균이 delta[m]만큼 증가)
# =========================
U = torch.randn(M, D, device=device)
U = U / (U.norm(dim=1, keepdim=True) + 1e-8)  # [M,D] unit

delta = torch.arange(M, device=device).float() * gap_step  # 0,2000,4000,...
c = torch.sqrt(delta + 1e-8)                               # ||shift|| = sqrt(delta)

tgt_feat = tgt_base.clone()
for b in range(B):
    for m in range(M):
        shift = c[m] * U[m].view(1, 1, D)     # [1,1,D]
        tgt_feat[b, m] = tgt_feat[b, m] + shift

    # 정답 후보는 추가로 "가깝게": dist_sq를 mstar_adv만큼 줄이는 방향으로 이동
    m0 = int(m_star[b])
    tgt_feat[b, m0] = tgt_feat[b, m0] - math.sqrt(mstar_adv) * U[m0].view(1,1,D)

# =========================
# 3) struct cost: 1 - attn (너 흐름 유지)
# =========================
def attn_cost_from_feat(feat, proj_dim=64):
    D_in = feat.shape[-1]
    wq = torch.randn(D_in, proj_dim, device=feat.device) / math.sqrt(D_in)
    wk = torch.randn(D_in, proj_dim, device=feat.device) / math.sqrt(D_in)
    Q = feat @ wq
    Kmat = feat @ wk
    scores = (Q @ Kmat.transpose(-2, -1)) / math.sqrt(proj_dim)
    attn = torch.softmax(scores, dim=-1)
    return 1.0 - attn  # cost

src_str = attn_cost_from_feat(src_feat)  # [B,N,N]
tgt_str = attn_cost_from_feat(tgt_feat.reshape(B*M, K, D)).reshape(B, M, K, K)

# =========================
# 4) flatten
# =========================
src_feat_exp = src_feat.unsqueeze(1).expand(B, M, N, D).reshape(B*M, N, D)
src_str_exp  = src_str.unsqueeze(1).expand(B, M, N, N).reshape(B*M, N, N)
tgt_feat_exp = tgt_feat.reshape(B*M, K, D)
tgt_str_exp  = tgt_str.reshape(B*M, K, K)

a = torch.ones(B*M, N, device=device) / N
b = torch.ones(B*M, K, device=device) / K

# =========================
# 5) dist_sq / dist_norm + 후보 gap 확인
# =========================
dist_sq = torch.cdist(src_feat_exp, tgt_feat_exp, p=2) ** 2  # [B*M,N,K]
dist_norm = dist_sq / float(D)

@torch.no_grad()
def qprint(x, name):
    x = x.detach().float().flatten()
    qs = torch.quantile(x, torch.tensor([0.0,0.5,0.9,0.95,0.99,1.0], device=x.device))
    print(f"\n[{name}] mean={x.mean().item():.1f} std={x.std().item():.1f}")
    print(f"[{name}] q00={qs[0].item():.1f} q50={qs[1].item():.1f} q90={qs[2].item():.1f} "
          f"q95={qs[3].item():.1f} q99={qs[4].item():.1f} q100={qs[5].item():.1f}")

qprint(dist_sq, "dist_sq")
qprint(dist_norm, "dist_sq/D (=dist_norm)")

d_feat_sq = dist_sq.mean(dim=(1,2)).reshape(B, M)  # [B,M]
print("\n[FEATURE-ONLY mean dist_sq per candidate] (min5)")
for bb in range(B):
    vals = d_feat_sq[bb].detach().cpu()
    best = torch.topk(-vals, k=5).indices
    print(f"  b={bb} | m*={int(m_star[bb])}")
    for idx in best.tolist():
        print(f"    m={idx:2d} : {vals[idx].item():.1f}")
    top2 = torch.topk(-vals, k=2).values
    print(f"    gap(top1-top2)={float(top2[0]-top2[1]):.1f}")

# =========================
# 6) RAW vs SCALED(CF+CS mean-scaling) FGW
# =========================
M_feat_raw = dist_norm

sf = dist_norm.detach().mean() + 1e-8
M_feat_scaled = dist_norm / sf

ss = src_str_exp.detach().mean() + 1e-8
st = tgt_str_exp.detach().mean() + 1e-8
src_str_scaled = src_str_exp / ss
tgt_str_scaled = tgt_str_exp / st

def run_fgw(src_s, tgt_s, M_cost, tag):
    out = solve_gromov_batch(
        src_s, tgt_s, M=M_cost, alpha=alpha, reg=reg, a=a, b=b,
        max_iter=10, tol=1e-3, grad="envelope"
    )
    Tplan = out.plan.detach()
    val = out.value.detach()

    feat_term = (M_cost * Tplan).sum(dim=(1,2)).mean().item()
    total_val = val.mean().item()
    struct_term = (total_val - (1 - alpha) * feat_term) / (alpha + 1e-9)
    ratio = feat_term / (struct_term + 1e-9)

    print(f"\n[{tag}]")
    print(f"  M_cost mean={M_cost.mean().item():.4f}, std={M_cost.std().item():.4f}")
    print(f"  src_str mean={src_s.mean().item():.4f}, tgt_str mean={tgt_s.mean().item():.4f}")
    print(f"  FGW value mean={total_val:.6f}")
    print(f"  Feat term={feat_term:.6f} | Struct term={struct_term:.6f} | Ratio={ratio:.3f}")
    return out

out_raw = run_fgw(src_str_exp, tgt_str_exp, M_feat_raw, "RAW")
out_scl = run_fgw(src_str_scaled, tgt_str_scaled, M_feat_scaled, "SCALED (CF+CS mean-scaling)")

# =========================
# 7) align 체크
# =========================
d_raw = out_raw.value.reshape(B, M)
d_scl = out_scl.value.reshape(B, M)
pi_scl = torch.softmax(-d_scl / soft_tau, dim=1)

print("\n[ALIGN CHECK]")
for bb in range(B):
    raw_pick = int(torch.argmin(d_raw[bb]).item())
    scl_pick = int(torch.argmin(d_scl[bb]).item())
    print(f"  b={bb} | m*={int(m_star[bb])} | raw_pick={raw_pick} | scaled_pick={scl_pick} | "
          f"max_pi_scaled={pi_scl[bb].max().item():.4f}")
    print(f"    pi_scaled[{bb}]: {pi_scl[bb].detach().cpu().numpy().round(4)}")



[dist_sq] mean=38557.4 std=9760.8
[dist_sq] q00=21036.8 q50=38038.0 q90=52018.4 q95=53869.9 q99=56745.9 q100=59925.9

[dist_sq/D (=dist_norm)] mean=50.2 std=12.7
[dist_sq/D (=dist_norm)] q00=27.4 q50=49.5 q90=67.7 q95=70.1 q99=73.9 q100=78.0

[FEATURE-ONLY mean dist_sq per candidate] (min5)
  b=0 | m*=9
    m= 0 : 23811.9
    m= 1 : 26350.4
    m= 2 : 27657.2
    m= 3 : 29856.7
    m= 9 : 31891.3
    gap(top1-top2)=2538.5
  b=1 | m*=3
    m= 0 : 23955.0
    m= 3 : 25277.6
    m= 1 : 26103.2
    m= 2 : 28528.2
    m= 4 : 31419.4
    gap(top1-top2)=1322.6
  b=2 | m*=4
    m= 0 : 24431.8
    m= 1 : 26266.3
    m= 4 : 26521.8
    m= 2 : 28030.4
    m= 3 : 29858.7
    gap(top1-top2)=1834.5
  b=3 | m*=7
    m= 0 : 24002.0
    m= 1 : 25890.4
    m= 2 : 27580.4
    m= 7 : 29556.5
    m= 3 : 30105.9
    gap(top1-top2)=1888.5

[RAW]
  M_cost mean=50.2050, std=12.7094
  src_str mean=0.8750, tgt_str mean=0.8750
  FGW value mean=24.492109
  Feat term=48.794044 | Struct term=0.190174 | Ratio=256.57

## Coordinate Clustering

- H_sample norm histogram (want left)
    - 각 샘플의 entropy H(pi)를 최대 엔트로피 log M으로 나누어 normalize
    - 0에 가까우면 거의 one hot이고 
    - 1에 가까우면 uniform 
    - 현재 막대가 0.98 ~ 1.0 에 몰려 있어서 sample entropy 낮게와는 정반대

- top1 prob histogram (want right)
    - 각 샘플에서 max_m pi_m 가장 큰 값의 분포 
    - flat이면 top 1이 1/M

- top1 - top2 histogram 
    - 각 샘플에서 (1등 확률 - 2등 확률) 차이 
    - 결정의 확신도 
    - 샤프하면 top1이 크고, top2가 작아서 margin이 크게 나옴 
    - 애매하면 top 1 ~ top 2여서 margin이 0 근처 
    - 현재 0 ~ 0.03 근처여서 top1과 top2가 거의 비슷함.

< 결과 >
- Top 1 과 Top 2의 차이가 거의 없다면 이들의 cost 차이가 없음을 의미 